# Data Streaming

Example for the Data Streaming module. This example demonstrates how to record data from an instrument continuously for a given duration including saving the data to file.
The data is in a format that can be easily loaded into a Pandas data frame.

Requirements:

* LabOne version >= 26.01
* `zhinst` version >= 26.01
* 1 Instrument with at least 1 Demodulator


In [ ]:
import zhinst.core as zi
import time
import matplotlib.pyplot as plt
import pandas
from pathlib import Path

## General configuration

### Connection to the instrument


In [ ]:
### UHFLI, VHFLI, GHFLI, SHFLI
device_id = "dev13016"
device_interface = "1GbE" # "1GbE" or "USB"
data_server_host = "localhost"
data_server_port = 8004
api_level = 6

### MFLI, MFIA
# device_id = "dev4299"
# device_interface = "PCIe"
# data_server_host = f"mf-{device_id}"
# data_server_port = 8004
# api_level = 6

### HF2LI
# device_id = "dev878"
# device_interface = "USB"
# data_server_host = "10.42.3.238"
# data_server_port = 8005
# api_level = 1

### Connection
daq = zi.ziDAQServer(data_server_host, data_server_port, api_level)
daq.connectDevice(device_id, device_interface)

devtype = daq.getString(f"/{device_id}/features/devtype")
print(f"The API client is connected to {device_id.upper()} of type {devtype} via the data server with the following version:")
print(f"Client: {daq.version()}")
print(f"Server: {daq.getString('/zi/about/version')}")

    The API client is connected to DEV13016 of type VHFLI via the data server with the following version:
    Client: 26.01
    Server: 26.01

### Device settings


In [ ]:
data_rate_nominal = 2000
daq.set([
    (f"/{device_id}/demods/0/rate", data_rate_nominal),
    (f"/{device_id}/demods/0/enable", 1),
])
data_rate_actual = daq.getDouble(f"/{device_id}/demods/0/rate")
print(f"Actual data rate of demodulator: {data_rate_actual:.2f} Sa/s")

clockbase = daq.getDouble(f"/{device_id}/clockbase")
print(f"Sampling rate of device timestamp: {clockbase*1e-6:.1f} MSa/s")

    Actual data rate of demodulator: 1525.88 Sa/s
    Sampling rate of device timestamp: 2000.0 MSa/s

## Data Streaming module

### General configuration

#### Instantiation


In [ ]:
stream = daq.dataStreamingModule()

#### Signal paths to record


In [ ]:
signal_paths = [
    f"/{device_id}/demods/0/sample.x",
    f"/{device_id}/demods/0/sample.y",
    f"/{device_id}/demods/0/sample.r",
]
stream.subscribe(signal_paths)

#### Data saving settings


In [ ]:
stream.set("save/fileformat", "hdf5") # Possible formats are "csv", "mat" (MATLAB) or "hdf5"
stream.set("save/filename", f"example_data_streaming_module_{device_id}")
stream.set("save/directory", str(Path(".").absolute()))

### Finite Acquisition

#### Duration of acquisition


In [ ]:
total_duration_sec = 4.0    # Seconds
stream.set("duration", total_duration_sec)

#### Acquire signals and save data before reading


In [ ]:
timeout_sec = 1.5 * total_duration_sec
start_time = time.time()

# Acquisition
stream.execute()
while time.time() - start_time < timeout_sec:
    print(f"Progress: {stream.progress()[0]*100:.0f}%")
    if stream.finished():
        break
    time.sleep(1.0)
print(f"Actual duration of acquisition: {stream.getDouble('duration'):.2f} seconds")

# Save
stream.set("save/save", 1)
while stream.getInt("save/save"):
    time.sleep(0.1)
print("Saving to file is complete.")

# Read
data = stream.read()
print("Data is available for processing.")

    Progress: 0%
    Progress: 5%
    Progress: 29%
    Progress: 52%
    Progress: 81%
    Progress: 100%
    Actual duration of acquisition: 4.20 seconds
    Saving to file is complete.
    Data is available for processing.

#### Extract and plot signals


In [ ]:
ts = data[f"/{device_id}/timestamp"]
t = (ts - ts[0]) / clockbase

fig = plt.figure()
for signal_path in signal_paths:
    plt.plot(t, data[signal_path], label=signal_path)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (V)")
plt.legend()
plt.grid()
plt.show()

### Record in Pandas data frames 


In [ ]:
total_duration_sec = 5.0    # Seconds
stream.set("duration", total_duration_sec)

#### Save to file while reading


In [ ]:
stream.set("save/saveonread", 1)

In [ ]:
df = pandas.DataFrame()

timeout_sec = 1.5 * total_duration_sec
start_time = time.time()

# Acquisition, read and save
stream.execute()
while time.time() - start_time < timeout_sec:
    print(f"Progress: {stream.progress()[0]*100:.0f}%")
    if stream.finished():
        data = stream.read()
        df = pandas.concat([df, pandas.DataFrame(data)])
        break
    time.sleep(1.0)
    data = stream.read()
    df = pandas.concat([df, pandas.DataFrame(data)])

print(f"Actual duration of acquisition: {stream.getDouble('duration'):.2f} seconds")
df.head(10)

    Progress: 0%
    Progress: 14%
    Progress: 33%
    Progress: 57%
    Progress: 81%
    Progress: 100%
    Actual duration of acquisition: 4.20 seconds

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>/dev13016/demods/0/sample.r</th>
      <th>/dev13016/demods/0/sample.x</th>
      <th>/dev13016/demods/0/sample.y</th>
      <th>/dev13016/timestamp</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2.470375e-07</td>
      <td>-5.441863e-08</td>
      <td>-2.409692e-07</td>
      <td>1063201051484</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2.627563e-07</td>
      <td>-9.810374e-08</td>
      <td>-2.437551e-07</td>
      <td>1063202362204</td>
    </tr>
    <tr>
      <th>2</th>
      <td>5.560992e-07</td>
      <td>-4.788068e-07</td>
      <td>-2.828257e-07</td>
      <td>1063203672924</td>
    </tr>
    <tr>
      <th>3</th>
      <td>9.956160e-07</td>
      <td>-8.380790e-07</td>
      <td>-5.374707e-07</td>
      <td>1063204983644</td>
    </tr>
    <tr>
      <th>4</th>
      <td>1.616191e-06</td>
      <td>-1.247640e-06</td>
      <td>-1.027359e-06</td>
      <td>1063206294364</td>
    </tr>
    <tr>
      <th>5</th>
      <td>1.966732e-06</td>
      <td>-1.339488e-06</td>
      <td>-1.440071e-06</td>
      <td>1063207605084</td>
    </tr>
    <tr>
      <th>6</th>
      <td>1.749568e-06</td>
      <td>-8.477450e-07</td>
      <td>-1.530463e-06</td>
      <td>1063208915804</td>
    </tr>
    <tr>
      <th>7</th>
      <td>1.336723e-06</td>
      <td>-1.461289e-07</td>
      <td>-1.328712e-06</td>
      <td>1063210226524</td>
    </tr>
    <tr>
      <th>8</th>
      <td>1.219694e-06</td>
      <td>5.110322e-07</td>
      <td>-1.107475e-06</td>
      <td>1063211537244</td>
    </tr>
    <tr>
      <th>9</th>
      <td>1.138617e-06</td>
      <td>6.753653e-07</td>
      <td>-9.166950e-07</td>
      <td>1063212847964</td>
    </tr>
  </tbody>
</table>
</div>

### Endless acquisition 

#### Duration of acquisition


In [ ]:
stream.set("duration", 0)   # Set the duration to 0 for endless acquisition

#### Save to file without reading


In [ ]:
stream.set("save/saveonly", 1)

#### Acquisition


In [ ]:
total_duration_sec = 3.0    # Seconds
stream.execute()
time.sleep(total_duration_sec)
stream.finish()
print(f"Acquired signals are recorded in the saved file.")

    Acquired signals are recorded in the saved file.

## Tear down


In [ ]:
stream.clear()
daq.disconnect()
del stream, daq